In [ ]:
!pip3 uninstall python-git
!pip3 install GitPython

In [2]:
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains.question_answering import load_qa_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

import os
from git import Repo

In [3]:
repo_path = "./test_repo"

In [ ]:
git clone --depth 1 https://github.com/langchain-ai/langchain test_repo

In [4]:
loader = GenericLoader.from_filesystem(
    repo_path + "/libs/core/langchain_core/",
    glob="**/*",
    suffixes=[".py"],
    exclude=["**/non-utf-8-encoding.py"],
    parser=LanguageParser(language=Language.PYTHON, parser_threshold=500)
)

documents = loader.load()
len(documents)

658

In [5]:
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=2000,
    chunk_overlap=200
)

texts = python_splitter.split_documents(documents)
len(texts)

1854

In [ ]:
os.environ["OPENAI_API_KEY"]="YOUR_OPENAI_API_KEY"

In [7]:
db = Chroma.from_documents(texts, OpenAIEmbeddings(disallowed_special=()))

retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 8}
)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************eonK. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [ ]:
llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",
    max_tokens=1000,
)

In [8]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Você é um revisor de código experiente. Forneça informações detalhadas sobre a revisão do código e sugestões de melhorias baseadas no contexto fornecido abaixo:\n\n{context}",
    ),
    ("user", "{input}"),
])

document_chain = create_stuff_documents_chain(llm, prompt)

retrieval_chain = create_retrieval_chain(retriever, document_chain)

NameError: name 'llm' is not defined

In [9]:
response = retrieval_chain.invoke(
    {"input": "Você pode revisar e sugerir melhorias para o código de RunnableBinding"}
)

NameError: name 'retrieval_chain' is not defined

In [ ]:
print(response["answer"])